In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/annotated_cpu/checkpoints/post_cell_11.pickle

In [ ]:
%%RecordEvent
%%time
### cell 12 ###

### Optimized Code ###

# Set initial value for the first index
gap_length_init = np.full(len(train), np.nan)
gap_length_init[0] = 7
train['gap_length'] = gap_length_init

# Calculate gap length conditions
same_id = train['id'] == train['id'].shift(1)
gap_start_shift = train['discourse_start'].shift(1)
gap_conditions_one = (train['discourse_start'] - train['discourse_end'].shift(1) > 1) & same_id

new_essay_first_gap = (train['id'] != train['id'].shift(1)) & (train['discourse_start'] != 0)

gaps_one = train['discourse_start'] - train['discourse_end'].shift(1) - 2
gaps_two = train['discourse_start'] - 1

# Apply calculated gaps
train['gap_length'] = np.where(gap_conditions_one, gaps_one, train['gap_length'])
train['gap_length'] = np.where(new_essay_first_gap, gaps_two, train['gap_length'])

# Deal with the end gap of each essay
last_ones = train.drop_duplicates(subset='id', keep='last')

last_ones['gap_end_length'] = np.where(
    last_ones['discourse_end'] < last_ones['essay_len'],
    last_ones['essay_len'] - last_ones['discourse_end'],
    np.nan
)

cols_to_merge = ['id', 'discourse_id', 'gap_end_length']
train = train.merge(last_ones[cols_to_merge], on=['id', 'discourse_id'], how='left')

gap_fills = np.where(last_ones['essay_len'] - last_ones['discourse_end'] > 0, last_ones['essay_len'] - last_ones['discourse_end'], np.nan)
train['gap_end_length'] = train['gap_end_length'].fillna(gap_fills)

train.head()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten_cpu/q20_rewrite/checkpoints/post_cell_12_try_2.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/opt_cell_exec_info_12_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[12], f)


In [ ]:
opt_output = Out.get(4)